In [1]:
!pip install -q rioxarray geopandas osmnx folium rasterio mapclassify scipy pyproj

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.7 MB/s eta 0:00:00


In [2]:
# ======================================================================================
# FLOOD MAP v5 — ПОЛНЫЙ ПАЙПЛАЙН DSM → DTM → КАРТА ЗАТОПЛЕНИЯ
# Самодостаточный файл для Kaggle Notebook (30 ГБ RAM)
# ======================================================================================
#
# ЯЧЕЙКА 1 (установка):
#   !pip install -q rioxarray geopandas osmnx folium rasterio mapclassify scipy pyproj
#
# ЯЧЕЙКА 2: вставить этот файл целиком
#
# ПАЙПЛАЙН:
#   1. Вертикальная коррекция: EGM2008 (pyproj) + bias по контрольным точкам
#   2. Маска покрытия: отсечение зон вне стрипов (треугольные артефакты)
#   3. DSM → DTM: удаление зданий по OSM + морфологическая фильтрация
#   4. Бинарная маска затопления (2 м, тайлами)
#   5. Анализ связности (10 м, в Python)
#   6. Финальная маска + векторизация + карта

import os, gc, warnings
import numpy as np
import geopandas as gpd
import pandas as pd
import rioxarray
import rasterio
import folium
from rasterio.features import shapes, rasterize as rio_rasterize
from shapely.geometry import box, Point
from scipy import ndimage
from osgeo import gdal

warnings.filterwarnings("ignore")
gdal.UseExceptions()

WD = "/kaggle/working/"
os.makedirs(WD, exist_ok=True)


# ══════════════════════════════════════════════════════════════════════════════
# ПАРАМЕТРЫ
# ══════════════════════════════════════════════════════════════════════════════
FLOOD_LEVEL = 3.05  # м БС-77

# Исходный DEM (уже скачан)
INPUT_DSM = "/kaggle/input/notebooks/ivanesin/flood-map-2-0/dem_merged_reprojected.tif"
CRS_METRIC = "EPSG:32636"

# Уровни с водомерных постов
GAUGE_CSV = "/kaggle/input/datasets/ivanesin/gi-level/.csv"  # ← ПОМЕНЯЙТЕ ПУТЬ
STRIP_DATE = "2022-03-09"

# Контрольные точки (верифицированные)
#   (name, lat, lon, H_BS77_m, type)
#   type: "water" — пост на воде (H_BS77 берётся из CSV в момент съёмки)
#         "ground" — земля/асфальт/ВПП (H_BS77 фиксированная)
CONTROL_POINTS = [
    # ВПП аэродромов — плоские открытые площадки, ICAO-верифицированные
    ("Пулково ВПП (ULLI)",   59.8003,  30.2625,  23.77, "ground"),  # 78 ft
    ("Ржевка ВПП (ULSS)",    59.9800,  30.5856,  17.07, "ground"),  # 56 ft
    ("Левашово ВПП (ULLV)",  60.0867,  30.1933,  17.98, "ground"),  # 59 ft
    # Метеоплощадка на КЗС — репер
    ("КЗС метеоплощадка",    59.9878,  29.6967,   3.37, "ground"),
    # Водомерные посты — точки НА ВОДЕ
    ("ГИ (на воде)",         59.9284,  30.2701,   None, "water"),
    ("С1 ФЗ (на воде)",      59.9903,  29.6917,   None, "water"),
    ("С2 ФЗ (на воде)",      59.9903,  29.6917,   None, "water"),
]

WATER_SEEDS = [
    (30.235, 59.952), (30.275, 59.930), (30.327, 59.934),
    (30.365, 59.948), (30.435, 59.947), (30.260, 59.940),
    (30.500, 59.955),
]

CONN_RES = 10       # разрешение связности (м)
CLOSING_M = 20      # морф. закрытие (м)
SEED_R = 20         # радиус поиска seed (px @ CONN_RES)
MIN_AREA = 500      # мин. площадь полигона (м²)
SIMPLIFY = 10       # упрощение геометрии (м)


# ══════════════════════════════════════════════════════════════════════════════
# 1. ВЕРТИКАЛЬНАЯ КОРРЕКЦИЯ
# ══════════════════════════════════════════════════════════════════════════════
def egm2008_undulation(lat, lon):
    """Ундуляция EGM2008 через pyproj."""
    try:
        from pyproj import Transformer
        t = Transformer.from_crs("EPSG:4979", "EPSG:4326+3855", always_xy=False)
        _, _, h = t.transform(lat, lon, 0.0)
        return -h
    except Exception:
        return 16.5  # приближение для СПб


def read_dem_at_point(dem_path, lat, lon, crs, radius_px=10):
    """Читает медиану DEM в окне вокруг точки. Не загружает весь файл."""
    pt = gpd.GeoDataFrame({'geometry': [Point(lon, lat)]}, crs="EPSG:4326").to_crs(crs)
    px, py = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y

    ds = gdal.Open(dem_path)
    gt = ds.GetGeoTransform()
    c = int((px - gt[0]) / gt[1])
    r = int((py - gt[3]) / gt[5])

    x0 = max(0, c - radius_px)
    y0 = max(0, r - radius_px)
    xs = min(2 * radius_px, ds.RasterXSize - x0)
    ys = min(2 * radius_px, ds.RasterYSize - y0)

    if xs <= 0 or ys <= 0:
        ds = None
        return np.nan

    arr = ds.GetRasterBand(1).ReadAsArray(x0, y0, xs, ys).astype(float)
    nd = ds.GetRasterBand(1).GetNoDataValue()
    ds = None

    if nd is not None:
        arr[arr == nd] = np.nan
    valid = arr[~np.isnan(arr)]
    valid = valid[(valid > -200) & (valid < 200)]
    return float(np.median(valid)) if len(valid) > 0 else np.nan


def get_water_levels(csv_path, date_str, hour_start=15, hour_end=16):
    """Уровни с постов в момент съёмки (см → м, округлено до 0.01)."""
    levels = {}
    try:
        df = pd.read_csv(csv_path)
        df['_dt'] = pd.to_datetime(df.iloc[:, 0])
        day = df[df['_dt'].dt.strftime('%Y-%m-%d') == date_str]
        win = day[(day['_dt'].dt.hour >= hour_start) & (day['_dt'].dt.hour <= hour_end)]
        for col in ['ГИ', 'С1 ФЗ', 'С2 ФЗ']:
            v = pd.to_numeric(win[col], errors='coerce').dropna() / 100
            if not v.empty:
                levels[col] = round(float(v.mean()), 2)
    except Exception as e:
        print(f"   ⚠ CSV: {e}")
    return levels


def calibrate(dem_path, control_pts, csv_path, crs):
    """Вычисляет total_offset = EGM2008_N + bias."""
    print("   Контрольные точки:")
    print(f"   {'Точка':<28s} {'DSM':>7s} {'N_EGM':>7s} {'H_ort':>7s} {'H_BS77':>7s} {'Δ':>7s}")
    print(f"   {'-'*70}")

    wl = get_water_levels(csv_path, STRIP_DATE)
    if wl:
        print(f"   Уровни {STRIP_DATE}: " + ", ".join(f"{k}={v:.2f}" for k, v in wl.items()))

    biases = []
    egm_vals = []

    for name, lat, lon, h_known, ptype in control_pts:
        dsm_val = read_dem_at_point(dem_path, lat, lon, crs)
        n_egm = egm2008_undulation(lat, lon)
        egm_vals.append(n_egm)

        if np.isnan(dsm_val):
            print(f"   {name:<28s} {'N/A':>7s}")
            continue

        h_ortho = dsm_val - n_egm  # высота над EGM2008

        if ptype == "water":
            # ищем уровень из CSV
            col_map = {"ГИ (на воде)": "ГИ", "С1 ФЗ (на воде)": "С1 ФЗ", "С2 ФЗ (на воде)": "С2 ФЗ"}
            csv_col = col_map.get(name)
            h_bs77 = wl.get(csv_col) if csv_col else None
            if h_bs77 is None:
                h_bs77 = 0.0  # нуль БС-77 как fallback
        else:
            h_bs77 = h_known

        bias = h_ortho - h_bs77
        print(f"   {name:<28s} {dsm_val:>7.2f} {n_egm:>7.2f} {h_ortho:>7.2f} {h_bs77:>7.2f} {bias:>+7.2f}")

        # Для калибровки берём только наземные точки и водомерные
        # DSM-эффект (крыши) даёт bias >> 5 м → фильтруем
        if ptype == "water" or (ptype == "ground" and abs(bias) < 5.0):
            biases.append(bias)

    egm_mean = float(np.mean(egm_vals)) if egm_vals else 16.5
    bias_med = float(np.median(biases)) if biases else 0.0
    offset = egm_mean + bias_med

    print(f"\n   EGM2008 среднее:     {egm_mean:>+.2f} м")
    print(f"   Bias (медиана):      {bias_med:>+.2f} м")
    print(f"   ─────────────────────────────")
    print(f"   Итоговый offset:     {offset:>+.2f} м")
    print(f"   Порог затопления:    {FLOOD_LEVEL + offset:.2f} м (в DSM)")
    return offset


# ══════════════════════════════════════════════════════════════════════════════
# 2. МАСКА ПОКРЫТИЯ
# ══════════════════════════════════════════════════════════════════════════════
def create_coverage_mask(dsm_path, out_path, crs, res=10):
    """Маска покрытия DEM: 1 = есть данные, 0 = за пределами стрипов.
    Дыры на воде ВНУТРИ покрытия заполняются closing."""
    print("   Создание маски покрытия из nodata...")

    dsm_10m = os.path.join(WD, "dsm_10m_cov.tif")
    if not os.path.exists(dsm_10m):
        opts = gdal.WarpOptions(format="GTiff", xRes=res, yRes=res,
                                resampleAlg="bilinear", dstNodata=-9999,
                                creationOptions=["COMPRESS=LZW", "TILED=YES"])
        gdal.Warp(dsm_10m, dsm_path, options=opts)

    with rasterio.open(dsm_10m) as src:
        arr = src.read(1)
        nd = src.nodata
        mask = np.ones_like(arr, dtype=np.uint8)
        if nd is not None:
            mask[arr == nd] = 0
        mask[np.isnan(arr)] = 0
        profile = src.profile.copy()

    # Closing: заполняет дыры на воде, НЕ расширяет края покрытия
    struct = ndimage.generate_binary_structure(2, 2)
    mask_closed = ndimage.binary_closing(mask, structure=struct, iterations=5).astype(np.uint8)

    # Клиппинг по городу
    try:
        import osmnx as ox
        ox.settings.timeout = 60
        city = ox.geocode_to_gdf("Saint Petersburg, Russia").to_crs(crs)
        city_mask = rio_rasterize(
            [(g, 1) for g in city.geometry],
            out_shape=mask_closed.shape,
            transform=rasterio.transform.from_bounds(
                *rasterio.open(dsm_10m).bounds, mask_closed.shape[1], mask_closed.shape[0]),
            fill=0, dtype='uint8'
        )
        mask_closed = mask_closed & city_mask
        print("   + клиппинг по границе города")
    except Exception as e:
        print(f"   ⚠ Граница города недоступна: {e}")

    profile.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(mask_closed, 1)

    pct = 100 * np.sum(mask_closed) / mask_closed.size
    print(f"   Валидных: {np.sum(mask_closed):,} пикселей ({pct:.1f}%)")
    del arr, mask, mask_closed; gc.collect()
    return out_path


# ══════════════════════════════════════════════════════════════════════════════
# 3. DSM → DTM (метод дорожной сети — по Окуневой и др., МГУ 2025)
# ══════════════════════════════════════════════════════════════════════════════
# Идея: дороги = «истинная земля». Извлекаем высоты DSM вдоль дорог,
# интерполируем → DTM. Это убирает здания И деревья одновременно.

def build_dtm_from_roads(dsm_path, dtm_out_path, crs, res=10):
    """
    Строит DTM на основе высот вдоль дорожной сети:
    1. Загрузка дорог из OSM (highway=*)
    2. Сэмплинг DSM вдоль дорог (каждые 10 м)
    3. Фильтрация выбросов (мосты, эстакады)
    4. Интерполяция → DTM на сетке 10 м
    """
    import osmnx as ox
    from scipy.interpolate import griddata
    ox.settings.timeout = 120

    # --- Шаг 1: загрузка дорог ---
    print("   Загрузка дорожной сети из OSM...")
    dsm_10m = os.path.join(WD, "dsm_10m_cov.tif")
    if not os.path.exists(dsm_10m):
        opts = gdal.WarpOptions(format="GTiff", xRes=res, yRes=res,
                                resampleAlg="bilinear", dstNodata=-9999,
                                creationOptions=["COMPRESS=LZW", "TILED=YES"])
        gdal.Warp(dsm_10m, dsm_path, options=opts)

    with rasterio.open(dsm_10m) as src:
        dsm_arr = src.read(1)
        nd = src.nodata
        if nd is not None:
            dsm_arr[dsm_arr == nd] = np.nan
        transform = src.transform
        profile = src.profile.copy()
        bounds = src.bounds

    bbox_gdf = gpd.GeoDataFrame(
        {'geometry': [box(bounds.left, bounds.bottom, bounds.right, bounds.top)]},
        crs=crs
    )
    b = bbox_gdf.to_crs("EPSG:4326").total_bounds

    # Загружаем дороги чанками
    lat_step = 0.08
    all_roads = []
    lat = b[1]
    while lat < b[3]:
        lat_end = min(lat + lat_step, b[3])
        try:
            roads = ox.features_from_bbox(
                bbox=(b[0], lat, b[2], lat_end),
                tags={'highway': ['residential', 'tertiary', 'secondary', 'primary',
                                  'trunk', 'motorway', 'unclassified', 'service',
                                  'living_street', 'pedestrian']}
            )
            if not roads.empty:
                roads = roads[roads.geometry.type.isin(['LineString', 'MultiLineString'])]
                all_roads.append(roads[['geometry']])
                print(f"     {lat:.3f}–{lat_end:.3f}: {len(roads)} дорог")
        except Exception as e:
            print(f"     {lat:.3f}–{lat_end:.3f}: ⚠ {e}")
        lat = lat_end

    if not all_roads:
        print("   ⚠ Дороги не загружены. Используем DSM как есть.")
        return None

    roads_gdf = pd.concat(all_roads, ignore_index=True)
    roads_gdf = gpd.GeoDataFrame(roads_gdf, crs="EPSG:4326").to_crs(crs)
    print(f"   Всего дорог: {len(roads_gdf)}")

    # --- Шаг 2: сэмплинг высот вдоль дорог ---
    print("   Сэмплинг DSM вдоль дорог...")
    sample_step = res  # каждые 10 м
    pts_x, pts_y, pts_z = [], [], []

    for geom in roads_gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        # Для MultiLineString обрабатываем каждую линию
        lines = [geom] if geom.geom_type == 'LineString' else list(geom.geoms)
        for line in lines:
            length = line.length
            dist = 0
            while dist <= length:
                pt = line.interpolate(dist)
                # Пиксель в DSM
                c = int((pt.x - transform.c) / transform.a)
                r = int((pt.y - transform.f) / transform.e)
                if 0 <= r < dsm_arr.shape[0] and 0 <= c < dsm_arr.shape[1]:
                    val = dsm_arr[r, c]
                    if not np.isnan(val):
                        pts_x.append(pt.x)
                        pts_y.append(pt.y)
                        pts_z.append(val)
                dist += sample_step

    pts_x = np.array(pts_x)
    pts_y = np.array(pts_y)
    pts_z = np.array(pts_z)
    print(f"   Точек сэмплирования: {len(pts_z):,}")

    # --- Шаг 3: фильтрация выбросов (мосты, эстакады) ---
    # Мосты/эстакады дают аномально высокие значения на фоне окружения
    # Используем скользящий перцентиль: точка с z > P90 в окрестности 200 м — выброс
    # Упрощённая версия: глобальный фильтр по перцентилям
    if len(pts_z) > 100:
        p10, p90 = np.percentile(pts_z, [10, 90])
        # Допускаем разумный диапазон (±2σ от медианы)
        med = np.median(pts_z)
        std = np.std(pts_z)
        lo, hi = med - 3 * std, med + 2 * std  # Асимметрично: мосты сверху
        good = (pts_z >= lo) & (pts_z <= hi)
        n_before = len(pts_z)
        pts_x, pts_y, pts_z = pts_x[good], pts_y[good], pts_z[good]
        print(f"   После фильтрации мостов: {len(pts_z):,} (убрано {n_before-len(pts_z)})")

    if len(pts_z) < 1000:
        print("   ⚠ Мало точек для интерполяции. Используем DSM.")
        return None

    # --- Шаг 4: интерполяция → DTM ---
    print("   Интерполяция DTM (может занять 1-2 мин)...")
    # Сетка пикселей DSM
    rows, cols = dsm_arr.shape
    yi = np.arange(rows) * transform.e + transform.f + transform.e / 2
    xi = np.arange(cols) * transform.a + transform.c + transform.a / 2
    XI, YI = np.meshgrid(xi, yi)

    # Интерполяция nearest (быстро) + линейная для покрытых зон
    dtm = griddata(
        (pts_x, pts_y), pts_z, (XI, YI),
        method='nearest'  # nearest гарантирует заполнение всех пикселей
    )

    # Сглаживание для устранения артефактов дискретизации
    from scipy.ndimage import uniform_filter
    dtm = uniform_filter(dtm.astype(np.float32), size=5)

    # Ограничение: DTM не может быть выше DSM
    valid_dsm = ~np.isnan(dsm_arr)
    dtm[valid_dsm] = np.minimum(dtm[valid_dsm], dsm_arr[valid_dsm])

    print(f"   DTM: мин={np.nanmin(dtm):.1f}, макс={np.nanmax(dtm):.1f}, "
          f"медиана={np.nanmedian(dtm):.1f}")

    # Сохраняем
    profile.update(dtype='float32', count=1, nodata=-9999)
    with rasterio.open(dtm_out_path, 'w', **profile) as dst:
        dst.write(dtm.astype(np.float32), 1)

    del dsm_arr, dtm, XI, YI, pts_x, pts_y, pts_z; gc.collect()
    print(f"   DTM сохранён: {dtm_out_path}")
    return dtm_out_path


# ══════════════════════════════════════════════════════════════════════════════
# 4. БИНАРНАЯ МАСКА ЗАТОПЛЕНИЯ (тайлами)
# ══════════════════════════════════════════════════════════════════════════════
def threshold_mask_tiled(dsm_path, out_path, threshold, water_level, tile_sz=4096):
    """
    Тайловая пороговая маска:
    - nodata → заполняется water_level (связность через реки)
    - DSM ≤ threshold → 1
    Здания и деревья корректируются на шаге 5 через DTM.
    """
    src = gdal.Open(dsm_path)
    cols, rows = src.RasterXSize, src.RasterYSize
    band = src.GetRasterBand(1)
    src_nd = band.GetNoDataValue()

    drv = gdal.GetDriverByName('GTiff')
    dst = drv.Create(out_path, cols, rows, 1, gdal.GDT_Byte,
                     ['COMPRESS=LZW', 'TILED=YES', 'BLOCKXSIZE=256', 'BLOCKYSIZE=256'])
    dst.SetGeoTransform(src.GetGeoTransform())
    dst.SetProjection(src.GetProjection())
    dst.GetRasterBand(1).SetNoDataValue(255)

    n_ty = (rows + tile_sz - 1) // tile_sz

    for ty in range(n_ty):
        y_off = ty * tile_sz
        y_sz = min(tile_sz, rows - y_off)
        for tx in range((cols + tile_sz - 1) // tile_sz):
            x_off = tx * tile_sz
            x_sz = min(tile_sz, cols - x_off)

            tile = band.ReadAsArray(x_off, y_off, x_sz, y_sz).astype(np.float32)
            nd_mask = np.isnan(tile)
            if src_nd is not None:
                nd_mask |= (tile == src_nd)
            tile[nd_mask] = water_level

            mask = (tile <= threshold).astype(np.uint8)
            dst.GetRasterBand(1).WriteArray(mask, x_off, y_off)

        if (ty + 1) % max(1, n_ty // 5) == 0 or ty == n_ty - 1:
            print(f"     {100*(ty+1)/n_ty:.0f}%")

    dst.FlushCache()
    dst = None
    src = None


# ══════════════════════════════════════════════════════════════════════════════
# 5. СВЯЗНОСТЬ + ЗДАНИЯ НА 10 М
# ══════════════════════════════════════════════════════════════════════════════
def connectivity_analysis(mask_2m, out_conn, seeds, crs,
                          closing_m, conn_res, seed_r,
                          coverage_mask_path=None, dtm_path=None,
                          threshold=None):
    """Связность на пониженном разрешении + DTM-коррекция."""

    # Downsample маску
    mask_10m = os.path.join(WD, "mask_10m.tif")
    opts = gdal.WarpOptions(format="GTiff", xRes=conn_res, yRes=conn_res,
                            resampleAlg="max",
                            creationOptions=["COMPRESS=LZW", "TILED=YES"])
    gdal.Warp(mask_10m, mask_2m, options=opts)

    with rasterio.open(mask_10m) as src:
        arr = src.read(1)
        transform = src.transform
        profile = src.profile.copy()

    print(f"   Маска 10 м: {arr.shape} ({arr.shape[0]*arr.shape[1]:,} px)")

    # DSM→DTM коррекция: если есть DTM, применяем порог к DTM вместо DSM
    # Это заменяет DSM-маску на DTM-маску (убирает здания И деревья)
    if dtm_path and os.path.exists(dtm_path) and threshold is not None:
        print("   Применяем DTM-коррекцию (метод дорожной сети)...")
        with rasterio.open(dtm_path) as dsrc:
            dtm = dsrc.read(1)
        nd = -9999
        dtm[dtm == nd] = np.nan
        
        # DTM-маска: земля ниже порога → затоплено
        # (DSM-маска: поверхность ниже порога → затоплено, но крыши и деревья «сухие»)
        dtm_mask = (dtm <= threshold).astype(np.uint8)
        dtm_mask[np.isnan(dtm)] = 0
        
        # Nodata в исходном DEM (вода) → тоже затоплено
        # (в DSM-маске nodata уже заполнено уровнем воды)
        
        # Объединяем: DSM-маска OR DTM-маска
        # DSM ловит воду (nodata→заполнено), DTM ловит застройку и деревья
        n_dsm_only = int(np.sum(arr & ~dtm_mask))
        n_dtm_only = int(np.sum(~arr & dtm_mask))
        arr = arr | dtm_mask
        
        print(f"   + DTM добавил {n_dtm_only:,} px (застройка + растительность)")
        print(f"   DSM-only (вода/nodata): {n_dsm_only:,} px")
        del dtm, dtm_mask

    # Closing
    closing_px = max(1, int(round(closing_m / conn_res)))
    print(f"   Closing: {closing_m} м → {closing_px} px")
    struct = ndimage.generate_binary_structure(2, 2)
    arr = ndimage.binary_closing(arr, structure=struct, iterations=closing_px).astype(np.uint8)

    # Connected components
    labels, n_feat = ndimage.label(arr)
    print(f"   Компонент: {n_feat}")

    # Seeds
    valid_labels = set()
    for lon, lat in seeds:
        pt = gpd.GeoDataFrame({'geometry': [Point(lon, lat)]}, crs="EPSG:4326").to_crs(crs)
        sx, sy = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
        c = int((sx - transform.c) / transform.a)
        r = int((sy - transform.f) / transform.e)
        r0, r1 = max(0, r-seed_r), min(labels.shape[0], r+seed_r+1)
        c0, c1 = max(0, c-seed_r), min(labels.shape[1], c+seed_r+1)
        found = set(np.unique(labels[r0:r1, c0:c1])) - {0}
        if found:
            valid_labels.update(found)
            print(f"   Seed ({lon:.3f},{lat:.3f}) → {sorted(found)[:3]}")
        else:
            print(f"   ⚠ Seed ({lon:.3f},{lat:.3f}) — нет маски в {seed_r*conn_res} м")

    if not valid_labels:
        print("   ⚠ Fallback: крупные компоненты")
        for lbl in range(1, n_feat + 1):
            if np.sum(labels == lbl) > 5000:
                valid_labels.add(lbl)

    conn = np.isin(labels, list(valid_labels)).astype(np.uint8)

    # Маска покрытия
    if coverage_mask_path and os.path.exists(coverage_mask_path):
        with rasterio.open(coverage_mask_path) as csrc:
            cov = csrc.read(1)
        if cov.shape != conn.shape:
            from scipy.ndimage import zoom
            cov = zoom(cov, (conn.shape[0]/cov.shape[0], conn.shape[1]/cov.shape[1]), order=0)
            cov = cov[:conn.shape[0], :conn.shape[1]]
        n_before = int(np.sum(conn))
        conn = conn & cov
        print(f"   Покрытие: удалено {n_before - int(np.sum(conn)):,} px вне зоны")

    n_final = int(np.sum(conn))
    area_ha = n_final * (conn_res ** 2) / 10000
    print(f"   Итого: {n_final:,} px (~{area_ha:.0f} га)")

    profile.update(dtype='uint8', nodata=255)
    with rasterio.open(out_conn, 'w', **profile) as dst:
        dst.write(conn, 1)

    del labels, arr, conn; gc.collect()
    return out_conn


# ══════════════════════════════════════════════════════════════════════════════
# 6. ФИНАЛЬНАЯ МАСКА 2 М + КАРТА
# ══════════════════════════════════════════════════════════════════════════════
def apply_filter_2m(mask_2m, conn_10m, out_final, tile_sz=4096):
    """Upsample фильтр → AND с маской 2 м (тайлами)."""
    conn_2m = os.path.join(WD, "conn_2m.tif")

    src_mask = gdal.Open(mask_2m)
    gt = src_mask.GetGeoTransform()

    opts = gdal.WarpOptions(format="GTiff",
                            xRes=abs(gt[1]), yRes=abs(gt[5]),
                            resampleAlg="near",
                            creationOptions=["COMPRESS=LZW", "TILED=YES"])
    gdal.Warp(conn_2m, conn_10m, options=opts)

    src_filt = gdal.Open(conn_2m)
    cols, rows = src_mask.RasterXSize, src_mask.RasterYSize

    drv = gdal.GetDriverByName('GTiff')
    dst = drv.Create(out_final, cols, rows, 1, gdal.GDT_Byte,
                     ['COMPRESS=LZW', 'TILED=YES'])
    dst.SetGeoTransform(gt)
    dst.SetProjection(src_mask.GetProjection())

    for ty in range((rows + tile_sz - 1) // tile_sz):
        y_off = ty * tile_sz
        y_sz = min(tile_sz, rows - y_off)
        for tx in range((cols + tile_sz - 1) // tile_sz):
            x_off = tx * tile_sz
            x_sz = min(tile_sz, cols - x_off)

            m = src_mask.GetRasterBand(1).ReadAsArray(x_off, y_off, x_sz, y_sz)
            fc, fr = src_filt.RasterXSize, src_filt.RasterYSize
            result = np.zeros_like(m)
            if x_off < fc and y_off < fr:
                fxs = min(x_sz, fc - x_off)
                fys = min(y_sz, fr - y_off)
                f = src_filt.GetRasterBand(1).ReadAsArray(x_off, y_off, fxs, fys)
                result[:fys, :fxs] = m[:fys, :fxs] & f
            dst.GetRasterBand(1).WriteArray(result, x_off, y_off)

    dst.FlushCache()
    dst = None; src_mask = None; src_filt = None
    if os.path.exists(conn_2m):
        os.remove(conn_2m)


def vectorize_and_map(final_mask, crs, out_html):
    """Векторизация + Folium карта."""
    with rasterio.open(final_mask) as src:
        arr = src.read(1)
        transform = src.transform
        bounds = src.bounds

    print(f"   Затопленных пикселей: {int(np.sum(arr)):,}")
    if np.sum(arr) == 0:
        print("   Затопления нет."); return None

    geoms = list({'properties': {'v': int(v)}, 'geometry': s}
                 for s, v in shapes(arr, mask=arr == 1, transform=transform))
    del arr; gc.collect()

    if not geoms:
        return None

    gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
    del geoms; gc.collect()

    gdf = gdf[gdf.geometry.area >= MIN_AREA]
    print(f"   Полигонов (≥{MIN_AREA} м²): {len(gdf)}")
    if gdf.empty:
        return None

    gdf = gdf.dissolve()
    gdf.geometry = gdf.geometry.simplify(SIMPLIFY)
    print(f"   Площадь: {gdf.geometry.area.sum()/10000:.0f} га")

    # Карта
    bbox_wgs = gpd.GeoDataFrame(
        {'geometry': [box(bounds.left, bounds.bottom, bounds.right, bounds.top)]},
        crs=crs
    ).to_crs("EPSG:4326").total_bounds

    cy, cx = (bbox_wgs[1]+bbox_wgs[3])/2, (bbox_wgs[0]+bbox_wgs[2])/2
    m = folium.Map(location=[cy, cx], zoom_start=11, tiles='CartoDB positron')

    flood_wgs = gdf.to_crs("EPSG:4326")
    folium.GeoJson(flood_wgs, name='Зона затопления',
                   style_function=lambda x: {'fillColor':'#1E90FF','color':'#0000CD',
                                             'weight':0.5,'fillOpacity':0.5}).add_to(m)

    # OSM landuse
    try:
        import osmnx as ox
        ox.settings.timeout = 60
        fb = flood_wgs.total_bounds
        lu = ox.features_from_bbox(bbox=(fb[0],fb[1],fb[2],fb[3]), tags={'landuse': True})
        if not lu.empty:
            lu = lu.to_crs(crs)
            lu = lu[lu.geometry.type.isin(['Polygon','MultiPolygon'])]
            impact = gpd.overlay(lu, gdf, how='intersection')
            impact['ha'] = impact.geometry.area / 10000
            print("\n   Landuse:")
            print(impact.groupby('landuse')['ha'].sum().sort_values(ascending=False).head(10).to_string())
            impact_wgs = impact.to_crs("EPSG:4326")
            folium.GeoJson(impact_wgs, name='Инфраструктура',
                           style_function=lambda x: {'fillColor':'#FF4500','color':'black',
                                                     'weight':0.5,'fillOpacity':0.5},
                           tooltip=folium.GeoJsonTooltip(
                               fields=['landuse','ha'], aliases=['Тип','Га'],
                               localize=True)).add_to(m)
    except Exception as e:
        print(f"   ⚠ OSM landuse: {e}")

    folium.LayerControl().add_to(m)
    m.save(out_html)
    print(f"   Карта: {out_html}")
    return m


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    print("=" * 70)
    print(f" FLOOD MAP v5 — DSM → DTM → КАРТА ЗАТОПЛЕНИЯ")
    print(f" Уровень: {FLOOD_LEVEL} м БС-77, разрешение: 2 м")
    print("=" * 70)

    # Пути промежуточных файлов
    mask_2m     = os.path.join(WD, "mask_2m.tif")
    conn_10m    = os.path.join(WD, "conn_10m.tif")
    final_2m    = os.path.join(WD, "flood_final_2m.tif")
    cov_mask    = os.path.join(WD, "coverage_mask.tif")
    dtm_10m     = os.path.join(WD, "dtm_roads_10m.tif")
    out_html    = os.path.join(WD, "spb_flood_v5.html")

    # 1. Калибровка
    print(f"\n[1/6] Вертикальная коррекция...")
    offset = calibrate(INPUT_DSM, CONTROL_POINTS, GAUGE_CSV, CRS_METRIC)
    threshold = FLOOD_LEVEL + offset
    water_level = offset  # 0 м БС-77 в координатах DSM

    # 2. Маска покрытия (10 м)
    print(f"\n[2/6] Маска покрытия + граница города...")
    create_coverage_mask(INPUT_DSM, cov_mask, CRS_METRIC)

    # 3. DTM из дорожной сети (10 м) — метод Окуневой и др. (МГУ, 2025)
    print(f"\n[3/6] DTM из дорожной сети (DSM→DTM)...")
    try:
        build_dtm_from_roads(INPUT_DSM, dtm_10m, CRS_METRIC)
    except Exception as e:
        print(f"   ⚠ DTM не построен: {e}")
        dtm_10m = None

    # 4. Пороговая маска 2 м (тайлами, ~0 RAM)
    print(f"\n[4/6] Бинарная маска 2 м (порог {threshold:.2f})...")
    threshold_mask_tiled(INPUT_DSM, mask_2m, threshold, water_level)

    # 5. Связность на 10 м + DTM-коррекция
    print(f"\n[5/6] Связность + DTM-коррекция...")
    connectivity_analysis(
        mask_2m, conn_10m, WATER_SEEDS, CRS_METRIC,
        CLOSING_M, CONN_RES, SEED_R,
        coverage_mask_path=cov_mask,
        dtm_path=dtm_10m if dtm_10m and os.path.exists(dtm_10m) else None,
        threshold=threshold
    )

    # 6. Финальная маска + карта
    print(f"\n[6/6] Финальная маска 2 м и карта...")
    apply_filter_2m(mask_2m, conn_10m, final_2m)

    # Очистка
    for f in [mask_2m, conn_10m]:
        if os.path.exists(f):
            os.remove(f)

    vectorize_and_map(final_2m, CRS_METRIC, out_html)

    print(f"\n{'='*70}")
    print(f" ГОТОВО!")
    print(f"{'='*70}")


main()

 FLOOD MAP v5 — DSM → DTM → КАРТА ЗАТОПЛЕНИЯ
 Уровень: 3.05 м БС-77, разрешение: 2 м

[1/6] Вертикальная коррекция...
   Контрольные точки:
   Точка                            DSM   N_EGM   H_ort  H_BS77       Δ
   ----------------------------------------------------------------------
   Уровни 2022-03-09: ГИ=0.17, С1 ФЗ=0.04, С2 ФЗ=0.03
   Пулково ВПП (ULLI)             36.07   -0.00   36.07   23.77  +12.30
   Ржевка ВПП (ULSS)              32.36   -0.00   32.36   17.07  +15.29
   Левашово ВПП (ULLV)            32.88   -0.00   32.88   17.98  +14.90
   КЗС метеоплощадка              16.67   -0.00   16.67    3.37  +13.30
   ГИ (на воде)                   16.26   -0.00   16.26    0.17  +16.09
   С1 ФЗ (на воде)                15.73   -0.00   15.73    0.04  +15.69
   С2 ФЗ (на воде)                15.73   -0.00   15.73    0.03  +15.70

   EGM2008 среднее:     +0.00 м
   Bias (медиана):      +15.70 м
   ─────────────────────────────
   Итоговый offset:     +15.70 м
   Порог затопления:    

In [3]:
# ======================================================================================
# КАЛИБРОВКА + КРОСС-СВЕРКА ArcticDEM ПО ВОДОМЕРНЫМ ПОСТАМ
# Запустите ПОСЛЕ шага [1/5] (merge_and_reproject), когда файл DEM готов
# ======================================================================================
#
# Результат: точное значение geoid_offset для подстановки в основной скрипт
#
# pip install rioxarray geopandas pandas numpy

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
from shapely.geometry import Point

# ==================== ПАРАМЕТРЫ ====================

# Файл DEM (результат merge_and_reproject)
# ВАЖНО: используем 10м версию для валидации (2м не помещается в RAM дважды)
# Если 10м файла нет, скрипт создаст его из 2м через GDAL Warp
DEM_FILE_2M = "/kaggle/input/notebooks/ivanesin/flood-map-2-0/dem_merged_reprojected.tif"
DEM_FILE_10M = "/kaggle/working/dem_validation_10m.tif"
CRS_METRIC = "EPSG:32636"

# CSV с уровнями
GAUGE_CSV = "/kaggle/input/datasets/ivanesin/gi-level/.csv"  # ← ПОМЕНЯЙТЕ ПУТЬ
DATETIME_COL = "Дата и время (московское зимнее)"

# Дата и время съёмки ArcticDEM стрипов
STRIP_DATE = "2022-03-09"
STRIP_UTC_HOUR = 12   # 12:40 UTC
STRIP_MSK_HOUR_START = 15  # 15:00 MSK (зимнее)
STRIP_MSK_HOUR_END = 16    # 16:59 MSK

# Водомерные посты: (name, lat, lon, csv_column)
GAUGES = [
    {"name": "ГИ (Горный институт)", "lat": 59.928441, "lon": 30.270109, "col": "ГИ",     "primary": True},
    {"name": "С1 ФЗ (Финский залив)","lat": 59.990286, "lon": 29.691681, "col": "С1 ФЗ",  "primary": False},
    {"name": "С2 ФЗ (Финский залив)","lat": 59.990286, "lon": 29.691681, "col": "С2 ФЗ",  "primary": False},
]

# Твёрдые точки на суше для кросс-сверки
# (name, lat, lon, known_elevation_BS77_m, source)
# Отметки взяты из открытых источников / топокарт
HARD_POINTS = [
    # Точки на набережных (высота гранитного парапета ≈ 2.5-3.5 м БС-77)
    ("Дворцовая наб. у Эрмитажа",   59.9408, 30.3135, 3.0, "топокарта"),
    ("Университетская наб.",         59.9375, 30.2975, 3.2, "топокарта"),
    # Известные площади/перекрёстки (плоские поверхности)
    ("Дворцовая площадь (центр)",    59.9390, 30.3150, 5.0, "топокарта"),
    ("Сенная площадь",               59.9270, 30.3200, 5.5, "топокарта"),
    # Железнодорожные станции (проверяемые отметки)
    ("Московский вокзал (перрон)",   59.9300, 30.3620, 6.0, "ж/д проф."),
    ("Балтийский вокзал (перрон)",   59.9070, 30.2990, 4.5, "ж/д проф."),
]


# ==================== ФУНКЦИИ ====================

def extract_dem_at_point(dem_arr, transform, lon, lat, crs_metric, search_r=10):
    """Извлекает значение DEM в точке (lon, lat) с поиском в окне."""
    pt = gpd.GeoDataFrame(
        {'geometry': [Point(lon, lat)]}, crs="EPSG:4326"
    ).to_crs(crs_metric)
    px, py = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
    
    col = int((px - transform.c) / transform.a)
    row = int((py - transform.f) / transform.e)
    
    # Ищем ближайшее валидное значение в окне
    best_val = np.nan
    best_dist = search_r + 1
    
    for dr in range(-search_r, search_r + 1):
        for dc in range(-search_r, search_r + 1):
            r, c = row + dr, col + dc
            if 0 <= r < dem_arr.shape[0] and 0 <= c < dem_arr.shape[1]:
                v = dem_arr[r, c]
                if not np.isnan(v) and abs(v) < 200:
                    dist = abs(dr) + abs(dc)
                    if dist < best_dist:
                        best_val = v
                        best_dist = dist
    
    return best_val, row, col


def get_water_level_at_time(df, col_name, date_str, hour_start, hour_end):
    """Извлекает уровень воды в указанный момент времени."""
    day = df[df['_dt'].dt.strftime('%Y-%m-%d') == date_str]
    if day.empty:
        return np.nan, 0
    
    # Точный час
    window = day[(day['_dt'].dt.hour >= hour_start) & (day['_dt'].dt.hour <= hour_end)]
    vals = pd.to_numeric(window[col_name], errors='coerce').dropna() / 100  # см → м
    
    if vals.empty:
        # Fallback: среднесуточное
        vals = pd.to_numeric(day[col_name], errors='coerce').dropna() / 100
    
    # Округляем до 1 см — точность исходных данных
    return round(float(vals.mean()), 2), len(vals)


# ==================== ОСНОВНОЙ КОД ====================

print("=" * 70)
print(" КАЛИБРОВКА + КРОСС-СВЕРКА ArcticDEM")
print(f" Дата съёмки: {STRIP_DATE}, ~{STRIP_UTC_HOUR}:40 UTC")
print("=" * 70)

# 1. Подготовка DEM (понижение разрешения для экономии RAM)
print("\n[1] Подготовка DEM...")
import os
if not os.path.exists(DEM_FILE_10M):
    from osgeo import gdal
    gdal.UseExceptions()
    print(f"    Понижаем разрешение 2м → 10м для валидации...")
    warp_opts = gdal.WarpOptions(
        format="GTiff", xRes=10, yRes=10, 
        resampleAlg="bilinear", dstNodata=-9999,
        creationOptions=["COMPRESS=LZW", "TILED=YES"]
    )
    ds = gdal.Warp(DEM_FILE_10M, DEM_FILE_2M, options=warp_opts)
    ds = None
    print(f"    Сохранено: {DEM_FILE_10M}")
else:
    print(f"    Используем готовый: {DEM_FILE_10M}")

dem = rioxarray.open_rasterio(DEM_FILE_10M, masked=True)
dem_arr = dem.values[0]
transform = dem.rio.transform()
print(f"    Размер: {dem_arr.shape}, CRS: {dem.rio.crs}")

# 2. Загрузка уровней
print("\n[2] Загрузка уровней воды...")
df = pd.read_csv(GAUGE_CSV)
df['_dt'] = pd.to_datetime(df[DATETIME_COL])
print(f"    Записей: {len(df)}")

# 3. Калибровка по каждому посту
print("\n[3] Калибровка по водомерным постам")
print("-" * 70)
print(f"{'Пост':<25s} {'DEM (WGS84)':>10s} {'Уровень':>10s} {'Геоид':>10s} {'N':>4s}")
print("-" * 70)

geoid_estimates = []

for gauge in GAUGES:
    dem_val, row, col = extract_dem_at_point(
        dem_arr, transform, gauge["lon"], gauge["lat"], CRS_METRIC
    )
    
    wl, n = get_water_level_at_time(
        df, gauge["col"], STRIP_DATE, STRIP_MSK_HOUR_START, STRIP_MSK_HOUR_END
    )
    
    if np.isnan(dem_val):
        print(f"{gauge['name']:<25s} {'N/A':>10s} {wl:>10.2f} {'N/A':>10s} {n:>4d}")
        continue
    
    geoid = dem_val - wl
    marker = " ★" if gauge["primary"] else ""
    print(f"{gauge['name']:<25s} {dem_val:>10.2f} {wl:>10.2f} {geoid:>10.2f} {n:>4d}{marker}")
    geoid_estimates.append({
        "name": gauge["name"],
        "dem": dem_val,
        "wl": wl,
        "geoid": geoid,
        "primary": gauge["primary"]
    })

if not geoid_estimates:
    print("\n⚠ Ни один пост не попал в DEM!")
else:
    # Основной offset (по первичному посту)
    primary = [g for g in geoid_estimates if g["primary"]]
    if primary:
        GEOID_OFFSET = primary[0]["geoid"]
        print(f"\n  ★ Принятый geoid_offset (по ГИ): {GEOID_OFFSET:.2f} м")
    else:
        GEOID_OFFSET = np.mean([g["geoid"] for g in geoid_estimates])
        print(f"\n  Средний geoid_offset: {GEOID_OFFSET:.2f} м")
    
    # Разброс между постами
    if len(geoid_estimates) > 1:
        geoids = [g["geoid"] for g in geoid_estimates]
        print(f"  Разброс между постами: {max(geoids) - min(geoids):.2f} м")
        print(f"  (допустимо ≤ 0.5 м, т.к. геоид плавно меняется на ~50 км)")

# 4. Кросс-сверка по твёрдым точкам
print(f"\n\n[4] Кросс-сверка по твёрдым точкам на суше")
print("-" * 70)
print(f"{'Точка':<35s} {'DEM':>8s} {'Ожид.':>8s} {'Δ':>8s} {'Источник'}")
print("-" * 70)

errors = []
for name, lat, lon, known_elev, source in HARD_POINTS:
    dem_val, _, _ = extract_dem_at_point(dem_arr, transform, lon, lat, CRS_METRIC)
    
    if np.isnan(dem_val):
        print(f"{name:<35s} {'N/A':>8s} {known_elev:>8.1f} {'N/A':>8s} {source}")
        continue
    
    # Пересчитываем DEM в ортометрическую высоту
    dem_ortho = dem_val - GEOID_OFFSET
    delta = dem_ortho - known_elev
    errors.append(delta)
    
    ok = "✓" if abs(delta) < 3.0 else "⚠"
    print(f"{name:<35s} {dem_ortho:>8.2f} {known_elev:>8.1f} {delta:>+8.2f} {source} {ok}")

if errors:
    errors = np.array(errors)
    print("-" * 70)
    print(f"  Среднее отклонение:    {np.mean(errors):>+.2f} м")
    print(f"  Стд. откл.:           {np.std(errors):>.2f} м")
    print(f"  RMS (корень из СКО):   {np.sqrt(np.mean(errors**2)):>.2f} м")
    print(f"  Макс. |отклонение|:    {np.max(np.abs(errors)):>.2f} м")
    
    # Оценка качества
    rms = np.sqrt(np.mean(errors**2))
    if rms < 2.0:
        quality = "ОТЛИЧНО (RMS < 2 м)"
    elif rms < 5.0:
        quality = "ХОРОШО (RMS < 5 м)"
    elif rms < 10.0:
        quality = "ПРИЕМЛЕМО (RMS < 10 м)"
    else:
        quality = "ПЛОХО (RMS ≥ 10 м) — проверьте калибровку"
    
    print(f"\n  Качество калибровки: {quality}")
    
    # Рекомендация по систематическому сдвигу
    if abs(np.mean(errors)) > 1.0:
        print(f"\n  ⚠ Систематический сдвиг {np.mean(errors):+.2f} м")
        print(f"    Скорректированный offset: {GEOID_OFFSET + np.mean(errors):.2f} м")
        print(f"    (ArcticDEM — это DSM, крыши/деревья завышают высоту)")

# 5. Итоговая рекомендация
print(f"\n\n{'=' * 70}")
print(f" ИТОГ: ПАРАМЕТРЫ ДЛЯ ОСНОВНОГО СКРИПТА")
print(f"{'=' * 70}")
print(f"""
В файле floodmap_v2_complete.py замените калибровку на жёсткое значение:

    # Калибровка по г/п ГИ, дата съёмки {STRIP_DATE}
    # Уровень воды в момент съёмки: {primary[0]['wl']:.2f} м (БС-77)
    # DEM в точке поста: {primary[0]['dem']:.2f} м (WGS84)
    GEOID_OFFSET = {GEOID_OFFSET:.2f}  # м
    
    # Порог затопления в координатах DEM:
    # threshold = FLOOD_LEVEL + GEOID_OFFSET
    # Для 3.05 м: {3.05 + GEOID_OFFSET:.2f} м

Или используйте автоматическую калибровку calibrate_by_gauge()
из основного скрипта — она сделает то же самое.
""" if primary else "\nНе удалось рассчитать offset.\n")

 КАЛИБРОВКА + КРОСС-СВЕРКА ArcticDEM
 Дата съёмки: 2022-03-09, ~12:40 UTC

[1] Подготовка DEM...
    Понижаем разрешение 2м → 10м для валидации...
    Сохранено: /kaggle/working/dem_validation_10m.tif
    Размер: (6779, 10445), CRS: EPSG:32636

[2] Загрузка уровней воды...
    Записей: 84762

[3] Калибровка по водомерным постам
----------------------------------------------------------------------
Пост                      DEM (WGS84)    Уровень      Геоид    N
----------------------------------------------------------------------
ГИ (Горный институт)           16.27       0.17      16.10    8 ★
С1 ФЗ (Финский залив)          15.73       0.04      15.69    8
С2 ФЗ (Финский залив)          15.73       0.03      15.70    8

  ★ Принятый geoid_offset (по ГИ): 16.10 м
  Разброс между постами: 0.42 м
  (допустимо ≤ 0.5 м, т.к. геоид плавно меняется на ~50 км)


[4] Кросс-сверка по твёрдым точкам на суше
----------------------------------------------------------------------
Точка            